# **SCAPES**

Follow this notebook to train your own SCAPES model and run it locally or online using the GUI. This notebook explains how SCAPES works, step by step. If you only want to train with the default settings and generate sounds, use `tutorial_simplified.ipynb`.

This notebook covers the three SCAPES tasks end-to-end:
1. **Dataprep**: encode the audio with the codec and compute the semantic annotations (CLAP)
2. **Training**: train the flow model and its local encoder
3. **Inference**: reconstruct, semantically reconstruct, interpolate and generate audio

On Colab the first code cell installs everything. Locally, install the environment first (README, Installation) and open the notebook from inside the repository.

# **0. Preamble**

The first code cell sets up the environment, as in the simplified notebook. The second cell holds every path and option of this notebook.

In [ ]:
# Setup. On Colab: clone SCAPES (with its submodule), mount Drive and install it with the README steps.
# Locally: use the repository this notebook is in (the folder above it that holds SCAPES/pipeline.py).
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # branch cloned on Colab

if IN_COLAB:
    from google.colab import drive

    # clone once: the folder /content/SCAPES is kept if the cell runs again
    if not os.path.isdir("/content/SCAPES"):
        subprocess.run(["git", "clone", "--recurse-submodules", "--branch", BRANCH,
                        "https://github.com/cordutie/SCAPES.git", "/content/SCAPES"], check=True)
    os.chdir("/content/SCAPES")
    drive.mount("/content/drive")

    # the install steps of the README (install:env), without the virtual environment
    python_m = [sys.executable, "-m"]
    subprocess.run(python_m + ["pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run(python_m + ["pip", "install", "-r", "requirements.txt",
                               "--extra-index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    subprocess.run(python_m + ["pip", "install", "--no-deps", "msclap==1.3.4",
                               "magenta-rt @ git+https://github.com/magenta/magenta-realtime@694a545e4ba0b88bf1150137b129582166d3e07f"],
                   check=True)
    subprocess.run(python_m + ["pip", "install", "music2latent==0.1.8"], check=True)
    subprocess.run(python_m + ["pip", "install", "-e", "./external/audio_codec_wrappers"], check=True)
    subprocess.run([sys.executable, "external/audio_codec_wrappers/scripts/fetch_weights.py", "--codec", "spectrostream"],
                   check=True)
    REPO = os.getcwd()
else:
    # walk up from the working directory to the repository root
    REPO = os.getcwd()
    while not os.path.exists(os.path.join(REPO, "SCAPES", "pipeline.py")):
        parent = os.path.dirname(REPO)
        if parent == REPO:
            raise FileNotFoundError("Open this notebook from inside the SCAPES repository.")
        REPO = parent

# the repository must be importable as SCAPES and quickstart
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("SCAPES repository:", REPO)

## Parameters

Edit the placeholders in the next cell, or set the environment variables `SCAPES_<NAME>` (for example `SCAPES_WAV_PATH`) to run the notebook without editing it. The codec, the conditioning rate and the memory are in the config file (section 1.2).

In [ ]:
import os

from SCAPES.data.config_loader import load_training_config
from SCAPES.pipeline import model_folder

# Paths. Edit the placeholders, or set the environment variable SCAPES_<NAME> instead.
WAV_PATH = os.environ.get("SCAPES_WAV_PATH", "path/to/your/wavs")          # folder with your WAV files (linked, not copied)
BUILD_PATH = os.environ.get("SCAPES_BUILD_PATH", "path/to/build_folder")   # dataset/ and models/ are written here
CONFIG_PATH = os.environ.get("SCAPES_CONFIG_PATH", "path/to/config.gin")   # the config file (SCAPES/config/config.gin)
AUDIO_PATH = os.environ.get("SCAPES_AUDIO_PATH", "path/to/your/audio.wav")    # file for the reconstructions
AUDIO_PATH_2 = os.environ.get("SCAPES_AUDIO_PATH_2", "path/to/end.wav")       # second file, for the interpolation
SEMANTIC_FLAG = os.environ.get("SCAPES_SEMANTIC_FLAG", "gentle_rain")         # a flag of cherry_picking.csv (3.2.4)
SEED = int(os.environ.get("SCAPES_SEED", 0))

# Read from the config file: the model folder (build_path/models/<size>_mem<ms>), the ODE steps and the guidance
TRAINING = load_training_config(CONFIG_PATH)
MODEL_PATH = model_folder(BUILD_PATH, CONFIG_PATH)
NFE = TRAINING.nfe              # ODE steps (inference.nfe, 16 by default)
CFG_SCALE = TRAINING.cfg_scale  # classifier-free guidance (inference.cfg_scale, 1.0 = off)
print("Model folder:", MODEL_PATH)

## Timing check (about one minute on a GPU)

How fast this machine makes sound with the model size and the generation settings of the config file. The model has random weights (no training, no checkpoint): it makes 10 seconds of sound ten times and prints the average.

In [ ]:
# Timing check: random weights, ten runs of 10 s of sound, with the ODE steps and guidance of the config file.
import time

import torch

from SCAPES.data.config_loader import load_dataprep_config
from SCAPES.inference.FlowInference import FlowInference
from SCAPES.pipeline import units_for_seconds

dataprep = load_dataprep_config(CONFIG_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
geometry = {"conditioning_rate": dataprep.conditioning_rate,
            "conditioning_memory_seconds": dataprep.conditioning_memory_seconds}
engine = FlowInference.from_config(dataprep.codec_name, size=TRAINING.size, geometry=geometry, device=device)
units = units_for_seconds(engine.geom, 10)
random_sound = lambda: [torch.randn(1024) for _ in range(units)]  # one random CLAP-like vector per unit

engine.generate_soundscape(random_sound(), nfe=TRAINING.nfe, cfg_scale=TRAINING.cfg_scale)  # warm-up: not timed
wall = seconds = 0.0
for _ in range(10):
    start = time.perf_counter()
    audio = engine.generate_soundscape(random_sound(), nfe=TRAINING.nfe, cfg_scale=TRAINING.cfg_scale)
    if device == "cuda":
        torch.cuda.synchronize()
    wall += time.perf_counter() - start
    seconds += audio.shape[-1] / engine.sr
per_second = wall / seconds
gpu = torch.cuda.get_device_name(0) if device == "cuda" else "no GPU (CPU)"
print(f"{gpu}, {dataprep.codec_name}, model '{TRAINING.size}': {per_second:.2f} s to make 1 s of sound. Real time: {'yes' if per_second < 1.0 else 'no'}.")

## Check if your runtime has a GPU

SCAPES uses a transformer architecture that benefits significantly from GPU acceleration. The next cell reports whether PyTorch sees a GPU.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# **1. Dataset Preparation**

In this section we encode the audio with the codec into latent frames, and compute the CLAP semantic embeddings of each part of each file.

## 1.1. Folders and files

Three things, in three places:

- the **wav folder**: your WAV files, in any folder (subfolders are included). SCAPES links to it and does not copy it.
- the **config file**: one file. Copy `SCAPES/config/config.gin` (every setting, with comments) or `SCAPES/config/config_simplified.gin` (the minimum), and edit it. If you use cherry picking, copy `SCAPES/config/cherry_picking.csv` next to it and name it in the config.
- the **build folder**: where SCAPES writes. It holds the dataset and the models:

```
<build folder>/
├── dataset/                    dataprep output; raw/ is a link to the wav folder (no copy)
│   └── dataprep_specs.json     the dataprep settings the dataset was made with
└── models/
    ├── large_mem500/           model size large, conditioning memory 0.5 s
    │   ├── checkpoints/
    │   └── semantics/          the cherry-picked semantics (online GUI and generation)
    └── small_mem500/
```

## 1.2. The config file

The settings you choose, in the config file:

| key | meaning |
|---|---|
| `codec` | `encodec24` and `spectrostream` (autoregressive); `encodec48` and `music2latent` (atoms) |
| `conditioning_rate` | how often the semantic condition changes, in Hz; rounded to the nearest rate the codec offers |
| `conditioning_memory_seconds` | how many seconds of past audio the model attends to; the model folder name uses milliseconds |
| `model.size` | `petite`, `tiny`, `small`, `medium`, `large` or `extra_large`; the model folder name uses it |
| `cherry_picking` | the path of the cherry-picking CSV, or empty for none; relative paths start at the folder of the config file |

In autoregressive mode the model generates one block of latent frames after another. In atoms mode it generates one atom per step: a block of latent frames overlapped with the previous one. `docs/generation_modes.html` has the diagrams and the rates each codec offers.

The rest of the config file holds the developer settings (dataprep: `semantic.*`, `splits.*`, `seed`, `dataset.hop_atoms`; training and inference). Changing the codec, the conditioning rate or a dataprep setting needs a new build folder: the run refuses the dataset and names the key that differs. Changing the memory or the model size does not: it trains a new model folder in the same build folder.

## 1.3. Check the build folder

`prepare_build` (section 1.4) reads the wav folder and the config file. This cell says what it will do before it encodes anything: `new` (there is no dataset yet, dataprep will run), `reuse` (the dataset was made with the same config and wavs, dataprep is skipped), or it stops with the reason.

In [ ]:
from pathlib import Path

from SCAPES.data.config_loader import load_dataprep_config
from SCAPES.data.dataprep.specs import check_dataset, dataprep_specs

# the dataprep settings of the config and the wav files, compared with the dataset in the build folder
specs = dataprep_specs(load_dataprep_config(CONFIG_PATH), Path(WAV_PATH), Path(CONFIG_PATH))
print(check_dataset(Path(BUILD_PATH), Path(WAV_PATH), specs))

## 1.4. Encode the audio and compute the annotations

`prepare_build` links the wav folder into `dataset/raw`, encodes every file with the codec and computes the CLAP embeddings. It writes `dataprep_specs.json` last, so the dataset is complete when that file exists. If the dataset is reused, nothing runs.

In [ ]:
from SCAPES.pipeline import prepare_build

DATASET = prepare_build(WAV_PATH, BUILD_PATH, CONFIG_PATH)
print("Dataset:", DATASET)

In [ ]:
# Free the memory of the step before, without a session reset
import gc

import torch

gc.collect()
torch.cuda.empty_cache()

## 1.5. Look at the annotations (optional)

A two-dimensional plot of the CLAP embeddings of the dataset, one point per segment. PCA is quick. t-SNE can take a while on a large dataset.

In [ ]:
from SCAPES.data.config_loader import load_dataprep_config
from SCAPES.data.dataset import AtomSequenceDataset
from SCAPES.data.visualization import LatentSpaceExplorer

viz_config = load_dataprep_config(CONFIG_PATH)
viz_dataset = AtomSequenceDataset(DATASET, config=viz_config, requested_keys=["target_semantic", "index"],
                                  device="cpu")
explorer = LatentSpaceExplorer(viz_dataset, max_samples_per_file=100)
explorer.plot_semantic(method="pca")
explorer.plot_semantic(method="tsne")

# **2. Training**

Training trains the model into the build folder and writes its semantics folder for the online GUI from the cherry-picking CSV named in the config. If you do not plan to use the online GUI, leave `cherry_picking` empty in the config file.

## 2.1. Model size and training settings

Edit the config file, then run the parameters cell again, so that `MODEL_PATH` follows the new size and memory. The settings you are most likely to change:

```
model.size = "large"                # petite, tiny, small, medium, large or extra_large
conditioning_memory_seconds = 0.5   # in the model folder name: large_mem500
training.max_epochs = 400           # the run stops earlier when the validation loss stops improving (patience)
training.batch_size = "auto"        # the largest batch that fits the free GPU memory, or an integer
training.learning_rate = 1e-4
training.checkpoint_freq = 15       # a checkpoint every 15 epochs (0: only the best and the last)
# training.max_steps = 2000         # optional step budget, instead of epochs
```

In [ ]:
from SCAPES.pipeline import train_in_build

# trains the model into MODEL_PATH (models/<size>_mem<ms>) and writes its semantics folder from the cherry-picking CSV
train_in_build(WAV_PATH, BUILD_PATH, CONFIG_PATH)
print("Training complete!")

## 2.2. Save your model (optional)

Packs the model folder into a zip file next to it, to download or upload.

In [ ]:
import shutil
from pathlib import Path

model_dir = Path(MODEL_PATH).resolve()
zip_path = shutil.make_archive(str(model_dir), "zip", root_dir=model_dir.parent, base_dir=model_dir.name)
print("Created:", zip_path)

# **3. Inference**

There are two ways to run inference: the online GUI, or Python directly in this notebook.

## **3.1. Online Inference**

In order to do online inference, you will need to upload your model to a Hugging Face model repository. This can be done directly in the browser using a free account. Your model folder is `models/<size>_mem<ms>` in the build folder.

First, clean up your model checkpoints and keep only the one you prefer. If in doubt, use the ones called `best`. You will also need to create a markdown file called `info.md`. Write whatever you want in it, as this information will appear as the model description in the GUI.

```
<model>/
├── info.md
├── checkpoints/
│   ├── best_flow_model.pt
│   ├── best_local_encoder.pt
│   └── inference.gin
└── semantics/
    ├── mars_sounds.pt
    ├── ...
    └── semantics.csv
```

Once the structure is correct, create a Hugging Face account, look for the option to create a new model repository, and follow the instructions. This will generate a git repository. Then use the browser File Uploader to drag and drop your full model folder into it, fill in the commit message at the bottom of the page, and press Commit.

Once the upload is done, copy the link to your repository and paste it into the Add Models option in our online GUI available at: https://huggingface.co/spaces/cordutie/SCAPES-demo

Have fun!

## **3.2. Local Inference**

Load the trained model and generate audio in various modes. All model and architecture settings come from `<model_dir>/checkpoints/inference.gin` (written during training).

**Key parameters.** `NFE` (number of function evaluations of the ODE solver: higher is slower, usually better) and `CFG_SCALE` (classifier-free guidance, off at 1.0 by default: above 1 it applies to every step and every codec, and higher gives more typical generations) are read from the config file (`inference.nfe`, `inference.cfg_scale`) in the parameters cell. The other two are arguments of the calls below:
- `TF` (teacher forcing): `True` uses the full ground-truth context (reconstruction); `"partial"` uses 0.5 s of audio, then generates freely (semantic reconstruction); `False` is fully generative (no teacher forcing)
- `decode_method`: `"ola_smooth"` (overlap-add with smoothing, default) or `"ola_linear"`

In [ ]:
import torch

from SCAPES.inference.FlowInference import FlowInference

device = "cuda" if torch.cuda.is_available() else "cpu"

# checkpoint: 'best' (best validation), 'last' (final), or an epoch number
engine = FlowInference(model_dir=MODEL_PATH, device=device, verbose=True, checkpoint="best")
print(f"Loaded model from {MODEL_PATH}")

### 3.2.1. Full reconstruction (teacher-forced)

Reconstructs audio from its full latent representation. The model gets ground-truth atoms as context, so this measures the fidelity of the autoencoder.

In [ ]:
from pathlib import Path

from IPython.display import Audio, display

from SCAPES.inference.FlowInference import run_resynthesis_pipeline

duration = 10  # seconds of the file to use

audio_tensor = engine.load_audio_to_tensor(AUDIO_PATH)[:, :, :duration * 48000]
print("Original:", Path(AUDIO_PATH).stem)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(engine=engine, audio_path=AUDIO_PATH, duration=duration, play=True,
                                  TF=True, NFE=NFE, cfg_scale=CFG_SCALE)

### 3.2.2. Semantic reconstruction (partial TF)

Uses only the first 0.5 s to extract the CLAP embedding, then generates the rest from scratch. Same semantic content, different acoustic realization.

In [ ]:
from pathlib import Path

from IPython.display import Audio, display

from SCAPES.inference.FlowInference import run_resynthesis_pipeline

duration = 10  # seconds of the file to use

audio_tensor = engine.load_audio_to_tensor(AUDIO_PATH)[:, :, :duration * 48000]
print("Original:", Path(AUDIO_PATH).stem)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(engine=engine, audio_path=AUDIO_PATH, duration=duration, play=True,
                                  TF="partial", NFE=NFE, cfg_scale=CFG_SCALE)

### 3.2.3. Semantic interpolation

Interpolates between the semantic embeddings of two audio files. `stickyness` controls the sharpness of the transition (1.0 is linear, higher gives a sharper midpoint).

In [ ]:
from pathlib import Path

from IPython.display import Audio, display

from SCAPES.inference.FlowInference import run_interpolation_pipeline

for path, label in [(AUDIO_PATH, "Start"), (AUDIO_PATH_2, "End")]:
    t = engine.load_audio_to_tensor(path)[:, :, :5 * 48000]
    print(f"{label}: {Path(path).stem}")
    display(Audio(t.detach().cpu().numpy()[0], rate=48000))

output = run_interpolation_pipeline(engine=engine, audio_path_1=AUDIO_PATH, audio_path_2=AUDIO_PATH_2,
                                    timeline_size=200, stay_time=20, stickyness=3.0, play=True,
                                    NFE=NFE, cfg_scale=CFG_SCALE, context_static=False)

### 3.2.4. Generation from the semantics folder

`run_generation` writes audio from the CLAP vectors of one flag of `cherry_picking.csv` (`SEMANTIC_FLAG`), without any input audio. It needs the semantics folder that training wrote. The output has one unit of audio per stored vector of the flag, so its length follows the segment in the CSV, and the file is written to `MODEL_PATH/generated/generation.wav`.

The codec, the conditioning rate and the memory are fixed in the trained model (its `inference.gin`). Changing them in the config file does not change a trained model: train a new one (a new build folder when the codec or the rate changes).

In [ ]:
from pathlib import Path

import torch
from IPython.display import Audio, display

from SCAPES.pipeline import run_generation

# one CLAP vector per row of the flag's file; each row is one unit of audio
prompts = list(torch.load(Path(MODEL_PATH) / "semantics" / f"{SEMANTIC_FLAG}.pt"))
wav = run_generation(model_dir=MODEL_PATH, out_path=Path(MODEL_PATH) / "generated", prompts=prompts,
                     seed=SEED, nfe=NFE, cfg_scale=CFG_SCALE)
display(Audio(str(wav)))